In [ ]:
# Importing libraries and defining paths

from pathlib import Path
import shutil, yaml

EXPORT = Path("../datasets/raw_export")
COLLAPSED = Path("../datasets/collapsed")

In [ ]:
# Inspecting labels

for split in ["train", "valid"]:

    total_boxes = 0
    empty_files = 0
    class_ids = set()

    files = list((EXPORT / split / "labels").glob("*.txt"))

    for f in files:
        text = f.read_text().strip()

        if text == "":
            empty_files += 1
            continue

        lines = text.split("\n")
        total_boxes += len(lines)

        for line in lines:
            class_ids.add(line.split()[0])

    print(f"{split.capitalize()} Label Files: {len(files)}")
    print(f"{split.capitalize()} Empty Files: {empty_files}")
    print(f"{split.capitalize()} Boxes: {total_boxes}")
    print(f"{split.capitalize()} Classes: {class_ids}")

In [ ]:
# Collapsing into a single class, and renaming as we copy
#
# The export's names ("WhatsApp Image 2026-09-11 at 06-18-27_jpeg.rf.ot0pcZ76....jpeg")
# carry spaces, parentheses and Roboflow's provenance hash. We rename to
# train_0001 / valid_0001 here rather than by hand, so raw_export stays exactly
# as Roboflow produced it and re-running this cell always gives the same names.

for split in ["train", "valid"]:
    src_images = EXPORT / split / "images"
    src_labels = EXPORT / split / "labels"
    dst_split = COLLAPSED / split

    # collapsed/ is fully derived from raw_export, so we rebuild it from empty.
    # Without this, a re-run would copy the new names in ALONGSIDE the old ones
    # (shutil.copy2 adds, it never removes) and silently double the dataset.
    # It also clears labels.cache, which Ultralytics writes next to the split
    # and which still lists the previous filenames.
    if dst_split.exists():
        shutil.rmtree(dst_split)
    (dst_split / "images").mkdir(parents=True)
    (dst_split / "labels").mkdir(parents=True)

    # Driven by the images, not by images and labels in separate loops: the
    # stem is the ONLY thing pairing an image with its label in YOLO, so both
    # must be renamed to the same number. Numbering the two folders separately
    # would drift apart the moment an image had no .txt (a legal way to mark a
    # background image) and attach every later label to the wrong image.
    #
    # sorted() so the numbering is reproducible, and because it keeps the
    # renamed files in the same order as the originals.
    for index, image in enumerate(sorted(src_images.glob("*")), start=1):
        new_stem = f"{split}_{index:04d}"

        shutil.copy2(image, dst_split / "images" / f"{new_stem}{image.suffix}")

        label = src_labels / f"{image.stem}.txt"
        if not label.exists():
            # A missing .txt is a background image. Leave it missing, so the
            # collapsed copy records it the same way the export did.
            continue

        text = label.read_text().strip()

        if text == "":
            (dst_split / "labels" / f"{new_stem}.txt").write_text("")
            continue

        new_lines = []
        for line in text.split("\n"):
            parts = line.split()
            parts[0] = "0"
            new_lines.append(" ".join(parts))

        (dst_split / "labels" / f"{new_stem}.txt").write_text("\n".join(new_lines))

    print(f"{split.capitalize()}: copied {index} images as {split}_0001..{split}_{index:04d}")

config = {
    "path": str(COLLAPSED.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "nc": 1,
    "names": ["object"],
}

(COLLAPSED / "data.yaml").write_text(yaml.safe_dump(config))
print((COLLAPSED / "data.yaml").read_text())

In [ ]:
# Confirming collapse

for split in ["train", "valid"]:

    total_boxes = 0
    empty_files = 0
    class_ids = set()

    files = list((COLLAPSED / split / "labels").glob("*.txt"))

    for f in files:
        text = f.read_text().strip()

        if text == "":
            empty_files += 1
            continue

        lines = text.split("\n")
        total_boxes += len(lines)

        for line in lines:
            class_ids.add(line.split()[0])

    print(f"{split.capitalize()} Label Files: {len(files)}")
    print(f"{split.capitalize()} Empty Files: {empty_files}")
    print(f"{split.capitalize()} Boxes: {total_boxes}")
    print(f"{split.capitalize()} Classes: {class_ids}")